# 15 · 部署画像：吞吐、p50/p95/p99 与量化误差

模型结构图上的 FLOPs 不是车端延迟。真实部署要测 warm-up、batch、内存、runtime、p50/p95/p99 latency 和数值误差。本 notebook 用 NumPy 模拟一个线性层，比较 float32 与简单 int8 weight/activation quantization 的误差和时延。

学习目标：

- 区分参数量、MACs/FLOPs、吞吐和端到端 latency；
- 正确报告 p50、p95、p99，而不是只报平均值；
- 观察 batch size、矩阵宽度和量化对误差/时延的影响；
- 明确这个 toy benchmark 不能替代 TensorRT、ONNX Runtime 或车规硬件实测。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter
from ipywidgets import interact, IntSlider

rng = np.random.default_rng(71)

def quantize(x):
    scale = max(float(np.max(np.abs(x))) / 127.0, 1e-8)
    q = np.clip(np.round(x / scale), -127, 127).astype(np.int8)
    return q, scale

def run_inference(batch=8, width=256, output_dim=128, repeats=80):
    local = np.random.default_rng(71)
    x = local.normal(0, 1, size=(batch, width)).astype(np.float32)
    weight = local.normal(0, 0.1, size=(width, output_dim)).astype(np.float32)
    bias = local.normal(0, 0.01, size=output_dim).astype(np.float32)
    xq, xs = quantize(x)
    wq, ws = quantize(weight)
    float_times, int_times = [], []
    for _ in range(10):
        _ = x @ weight + bias
        _ = (xq.astype(np.int32) @ wq.astype(np.int32)).astype(np.float32) * xs * ws + bias
    for _ in range(repeats):
        start = perf_counter()
        float_out = x @ weight + bias
        float_times.append((perf_counter() - start) * 1e6)
        start = perf_counter()
        int_out = (xq.astype(np.int32) @ wq.astype(np.int32)).astype(np.float32) * xs * ws + bias
        int_times.append((perf_counter() - start) * 1e6)
    float_times = np.asarray(float_times)
    int_times = np.asarray(int_times)
    error = np.mean(np.abs(float_out - int_out))
    return {
        'float': float_times,
        'int8': int_times,
        'mae': float(error),
        'macs': batch * width * output_dim,
    }

result = run_inference()
for key in ['float', 'int8']:
    values = result[key]
    print(key, 'p50/p95/p99 us:', np.percentile(values, [50, 95, 99]).round(2))
print('MACs:', result['macs'], 'quantization MAE:', result['mae'])


In [ ]:
def show_profile(batch=8, width=256):
    result = run_inference(batch=batch, width=width)
    labels = ['float32', 'simulated int8']
    p50 = [np.percentile(result['float'], 50), np.percentile(result['int8'], 50)]
    p95 = [np.percentile(result['float'], 95), np.percentile(result['int8'], 95)]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].bar(labels, p50, label='p50')
    ax[0].bar(labels, p95, bottom=p50, alpha=0.45, label='p95-p50')
    ax[0].set_ylabel('latency / us')
    ax[0].set_title(f'MACs={result["macs"]:,}, MAE={result["mae"]:.5f}')
    ax[0].legend()
    ax[1].hist(result['float'], bins=20, alpha=0.55, label='float32')
    ax[1].hist(result['int8'], bins=20, alpha=0.55, label='int8')
    ax[1].set_xlabel('latency / us')
    ax[1].set_title('latency distribution')
    ax[1].legend()
    plt.tight_layout()
    plt.show()
    print('float32 p50/p95/p99:', np.percentile(result['float'], [50, 95, 99]).round(2))
    print('int8 p50/p95/p99:', np.percentile(result['int8'], [50, 95, 99]).round(2))

interact(
    show_profile,
    batch=IntSlider(min=1, max=32, step=1, value=8, description='batch'),
    width=IntSlider(min=64, max=512, step=64, value=256, description='width'),
);


### 练习：把 benchmark 写成可复现证据

- 对 batch=1、8、32 分别记录 p50/p95/p99；不要只比较一次运行。
- 改变 width，估算 MACs，并比较 MACs 与实际 latency 是否线性。
- 计算量化相对误差，观察极端 outlier 对 per-tensor scale 的影响。
- 增加 warm-up、固定线程数和 CPU/GPU/车端型号字段。
- 下一步用 ONNX Runtime 或 TensorRT 做真正的 kernel、memory 和 end-to-end benchmark。


In [ ]:
batches = [1, 4, 8, 16, 32]
p95_float, p95_int8 = [], []
for batch in batches:
    result = run_inference(batch=batch, width=256, repeats=40)
    p95_float.append(np.percentile(result['float'], 95))
    p95_int8.append(np.percentile(result['int8'], 95))
plt.plot(batches, p95_float, marker='o', label='float32 p95')
plt.plot(batches, p95_int8, marker='o', label='int8 p95')
plt.xlabel('batch size')
plt.ylabel('p95 latency / us')
plt.title('batch-size latency curve')
plt.legend()
plt.show()


## 完成标准

- 报告一个固定硬件/软件环境下的 p50、p95、p99。
- 同时报告 MACs、吞吐、数值误差和 batch size。
- 解释为什么 p99、memory transfer、线程调度和 kernel fusion 会影响车端体验。
- 把这个 toy benchmark 设计成一个可替换 backend 的接口，接入真实部署 runtime 后仍能复用评测表。
